# Plot Cell Entry Effects of RSV Polymorphisms

This notebook analyzes the cell entry functional effects of naturally occurring RSV F protein polymorphisms identified from sequence alignments.

In [ ]:
# Parameters - will be overridden by papermill
strain = "RSV-A"
variable_sites_file = "results/polymorphisms/RSV-A_variable_sites.csv"
cell_entry_file = "results/summaries/cell_entry.csv"
output_csv = "results/polymorphisms/RSV-A_polymorphisms_with_effects.csv"
output_html = "results/polymorphisms/RSV-A_polymorphism_effects.html"

In [ ]:
import pandas as pd
import altair as alt

_ = alt.data_transformers.disable_max_rows()

## Configuration

In [ ]:
print(f"Analyzing strain: {strain}")
print(f"Variable sites input: {variable_sites_file}")
print(f"Cell entry data: {cell_entry_file}")
print(f"Output CSV: {output_csv}")
print(f"Output HTML: {output_html}")

## Load data

In [ ]:
# Load variable sites
polymorphisms = pd.read_csv(variable_sites_file)
print(f"Loaded {len(polymorphisms)} {strain} polymorphisms")
print(f"At {polymorphisms['site'].nunique()} sites")
display(polymorphisms.head())

# Load cell entry effects
cell_entry = pd.read_csv(cell_entry_file)
print(f"\nLoaded cell entry data: {len(cell_entry)} mutations")
print(f"Sites range: {cell_entry['site'].min()} - {cell_entry['site'].max()}")
display(cell_entry.head())

## Merge polymorphisms with functional effects

In [ ]:
# Merge on site, wildtype, and mutant
with_effects = polymorphisms.merge(
    cell_entry[['site', 'wildtype', 'mutant', 'cell entry', 'sequential_site', 'region']],
    on=['site', 'wildtype', 'mutant'],
    how='inner'
)

print(f"Matched {len(with_effects)} {strain} polymorphisms with cell entry data")
print(f"At {with_effects['site'].nunique()} sites")
print(f"\nCoverage: {len(with_effects) / len(polymorphisms) * 100:.1f}% of polymorphisms have DMS data")

# Display summary
display(with_effects.head(10))

# Save merged data
with_effects.to_csv(output_csv, index=False)
print(f"\nSaved merged data to: {output_csv}")

## Summary statistics

In [ ]:
print(f"Cell Entry Effect Statistics for {strain} Polymorphisms:")
print("=" * 60)
print(f"Mean effect: {with_effects['cell entry'].mean():.3f}")
print(f"Median effect: {with_effects['cell entry'].median():.3f}")
print(f"Std deviation: {with_effects['cell entry'].std():.3f}")
print(f"\nEffect range: {with_effects['cell entry'].min():.3f} to {with_effects['cell entry'].max():.3f}")

# Count by effect category
print("\nEffect Categories:")
print(f"  Beneficial (> 0.5): {(with_effects['cell entry'] > 0.5).sum()} ({(with_effects['cell entry'] > 0.5).sum() / len(with_effects) * 100:.1f}%)")
print(f"  Neutral (-0.5 to 0.5): {((with_effects['cell entry'] >= -0.5) & (with_effects['cell entry'] <= 0.5)).sum()} ({((with_effects['cell entry'] >= -0.5) & (with_effects['cell entry'] <= 0.5)).sum() / len(with_effects) * 100:.1f}%)")
print(f"  Deleterious (< -0.5): {(with_effects['cell entry'] < -0.5).sum()} ({(with_effects['cell entry'] < -0.5).sum() / len(with_effects) * 100:.1f}%)")

# Most tolerated polymorphisms
print("\nMost Tolerated Polymorphisms (highest cell entry):")
display(with_effects.nlargest(10, 'cell entry')[['site', 'mutation_type', 'cell entry', 'mutation_count', 'region']])

# Most deleterious polymorphisms
print("\nMost Deleterious Polymorphisms (lowest cell entry):")
display(with_effects.nsmallest(10, 'cell entry')[['site', 'mutation_type', 'cell entry', 'mutation_count', 'region']])

## Create interactive plots with Altair

In [ ]:
# Plot 1: Distribution histogram
mean_effect = with_effects['cell entry'].mean()
median_effect = with_effects['cell entry'].median()

hist = alt.Chart(with_effects).mark_bar().encode(
    alt.X('cell entry:Q', bin=alt.Bin(maxbins=50), title='Cell Entry Effect'),
    alt.Y('count()', title='Number of Polymorphisms'),
    tooltip=['count()']
).properties(
    width=600,
    height=400,
    title=f'Distribution of Cell Entry Effects for {strain} Polymorphisms (n={len(with_effects)})'
)

# Add reference lines
rule_zero = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='red', strokeDash=[5, 5], size=2).encode(x='x:Q')
rule_mean = alt.Chart(pd.DataFrame({'x': [mean_effect]})).mark_rule(color='blue', strokeDash=[5, 5]).encode(x='x:Q')
rule_median = alt.Chart(pd.DataFrame({'x': [median_effect]})).mark_rule(color='green', strokeDash=[5, 5]).encode(x='x:Q')

chart1 = hist + rule_zero + rule_mean + rule_median

chart1

In [ ]:
# Plot 2: Scatter plot by position
chart2 = alt.Chart(with_effects).mark_circle(opacity=0.7).encode(
    x=alt.X('site:Q', title='Site Position'),
    y=alt.Y('cell entry:Q', title='Cell Entry Effect'),
    size=alt.Size('mutation_count:Q', title='Mutation Count', scale=alt.Scale(range=[20, 400])),
    color=alt.Color('region:N', title='Region'),
    tooltip=['site', 'mutation_type', 'wildtype', 'mutant', 'mutation_count', 'cell entry', 'region']
).properties(
    width=700,
    height=400,
    title=f'Cell Entry Effects of {strain} Polymorphisms by Position'
)

# Add zero line
zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='red', strokeDash=[5, 5]).encode(y='y:Q')

chart2 = chart2 + zero_line

chart2

In [ ]:
# Plot 3: Violin plot by region
# Create the base violin plot
violin = alt.Chart(with_effects).transform_density(
    'cell entry',
    as_=['cell entry', 'density'],
    groupby=['region']
).mark_area(orient='horizontal', opacity=0.6).encode(
    x=alt.X('density:Q', title='', axis=None, stack='center'),
    y=alt.Y('cell entry:Q', title='Cell Entry Effect'),
    color=alt.Color('region:N', title='Region')
)

# Create individual data points with jitter
points = alt.Chart(with_effects).mark_circle(
    size=30,
    opacity=0.6
).encode(
    x=alt.X('jitter:Q', title='', axis=None),
    y=alt.Y('cell entry:Q', title='Cell Entry Effect'),
    color=alt.Color('region:N', title='Region'),
    tooltip=['site', 'mutation_type', 'wildtype', 'mutant', 'mutation_count', 'cell entry', 'region']
).transform_calculate(
    # Add random jitter for x position
    jitter='random()'
)

# Create zero line (needs to repeat for each facet)
zero_line = alt.Chart(with_effects).mark_rule(
    color='red',
    strokeDash=[5, 5]
).encode(
    y=alt.datum(0)
)

# Layer violin, points, and zero line, apply properties, then facet
chart3 = alt.layer(violin, points, zero_line).properties(
    width=80,
    height=400
).facet(
    column=alt.Column('region:N', title='Region', sort=alt.EncodingSortField(field='cell entry', op='median', order='ascending'))
).properties(
    title=f'Distribution of Cell Entry Effects by Region ({strain})'
)

chart3

## Save combined chart to HTML

In [ ]:
# Combine all charts vertically
combined_chart = alt.vconcat(
    chart1,
    chart2,
    chart3
).properties(
    title=f'{strain} Polymorphism Cell Entry Effects Analysis'
).configure_view(
    strokeWidth=0
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=20,
    anchor='middle'
)

# Save to HTML
combined_chart.save(output_html)
print(f"Saved interactive plots to: {output_html}")

combined_chart